# Five Rodinia workloads: GPU cycles and transfer traces
Select **Runtime → Change runtime type → GPU**, then run the cells in order. This notebook builds and runs **BFS, LavaMD, Particle Filter, K-means and B+tree**, using the same small scenarios as the source-analysis graphs.

The primary results are **measured kernel cycles**, not durations converted into cycles:
- `sm__cycles_elapsed.avg`: elapsed cycles averaged across SM instances for each kernel launch.
- `sm__cycles_active.avg`: active cycles averaged across SM instances.

These are **not whole-application cycles**. CPU work and PCIe copies have no single shared SM clock domain. Summing kernel elapsed cycles is a kernel-only total, not end-to-end latency. Nsight Systems retains timestamps separately because they are needed to reconstruct transfer/order DAGs.

**Colab may deny hardware-counter access.** If Nsight Compute reports `ERR_NVGPUCTRPERM`, the bundle will contain the error and any available Systems traces, but no claimed cycle measurement. Having Nsight Compute on your laptop cannot recover counters that were not collected during execution. The same capture script can run on a Linux GPU host with profiling permission.

Each workload gets one Systems capture and one separate Compute capture; there is no repetition sweep. The profilers may internally replay kernels. These tiny cases validate graph structure, not representative benchmark performance.

References: [NVIDIA metric definitions and profiling behavior](https://docs.nvidia.com/nsight-compute/ProfilingGuide/) · [CLI options](https://docs.nvidia.com/nsight-compute/NsightComputeCli/) · [Counter permissions](https://developer.nvidia.com/ERR_NVGPUCTRPERM).


In [ ]:
from pathlib import Path
import datetime, json, subprocess, shutil, sys, re, os
RUN = Path('/content/rodinia_cycles') / datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
RUN.mkdir(parents=True)
LOGS = RUN / 'logs'; LOGS.mkdir()
def run(args, label, check=True):
    args = list(map(str,args))
    try:
        p = subprocess.run(args, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=600)
        output, rc = p.stdout, p.returncode
    except (OSError, subprocess.TimeoutExpired) as e:
        output, rc = str(e), -1
    (LOGS / (label + '.log')).write_text(output)
    print(output[-2000:])
    if check and rc: raise RuntimeError(f'{label} failed; retain the logs with the final download cell.')
    return rc, output
if not shutil.which('nvidia-smi') or not shutil.which('nvcc'):
    raise RuntimeError('GPU runtime not available. Select Runtime → Change runtime type → GPU, reconnect, then start again.')
run(['nvidia-smi'], 'gpu_check')
print('Capture directory:', RUN)


In [ ]:
BRANCH = 'pa2746-byte-sampleworkloads'
REPO = Path('/content/ESP_workload_cycles')
URL = 'https://github.com/pa2746-byte/ESP_workload.git'
if not REPO.exists():
    run(['git','clone','--depth','1','--branch',BRANCH,URL,REPO], 'clone')
else:
    _, remote = run(['git','-C',REPO,'remote','get-url','origin'], 'origin')
    if remote.strip() != URL: raise RuntimeError('Existing directory belongs to a different repository.')
    _, dirty = run(['git','-C',REPO,'status','--porcelain'], 'git_status')
    if dirty.strip(): raise RuntimeError('Existing checkout has changes; use a fresh REPO directory instead of overwriting them.')
    run(['git','-C',REPO,'pull','--ff-only','origin',BRANCH], 'pull')
_, revision = run(['git','-C',REPO,'rev-parse','HEAD'], 'revision')
(RUN / 'repository.json').write_text(json.dumps({'url':URL,'branch':BRANCH,'commit':revision.strip()},indent=2))
sys.dont_write_bytecode = True
sys.path.insert(0,str(REPO/'tools'))
import capture_rodinia_cycles as capture
import importlib
capture = importlib.reload(capture)


## Locate or install the collectors
The next cell first uses installed tools. If either tool is missing, it checks the runtime's existing APT repositories and installs an available NVIDIA package. Installation failures are saved; they do not turn an unprofiled run into a valid measurement. No driver permissions are changed.


In [ ]:
INSTALL_MISSING_TOOLS = True
if INSTALL_MISSING_TOOLS and (not capture.locate('nsys') or not capture.locate('ncu')):
    run(['apt-get','update','-qq'], 'apt_update', check=False)
    for tool, prefix in [('nsys','nsight-systems'), ('ncu','nsight-compute')]:
        if capture.locate(tool): continue
        _, listing = run(['apt-cache','search','--names-only','^'+prefix], tool+'_packages', check=False)
        names = [line.split(' - ',1)[0].strip() for line in listing.splitlines()]
        names = [n for n in names if re.fullmatch(re.escape(prefix)+r'(?:-cli)?(?:-\d[\d.]*)?',n)]
        preferred = prefix+'-cli' if tool=='nsys' else prefix
        if preferred in names: package = preferred
        elif names: package = max(names,key=lambda n: tuple(map(int,re.findall(r'\d+',n))) or (0,))
        else:
            print(f'No {tool} package is available in this runtime; capture will report it missing.')
            continue
        run(['apt-get','install','-y',package], tool+'_install', check=False)
print('Nsight Systems:', capture.locate('nsys'))
print('Nsight Compute:', capture.locate('ncu'))


## Build and collect
Compatibility edits are made only in the staged copies and recorded in each `build_manifest.json`: legacy synchronization API names; K-means device 1 → 0, modern symbol-copy syntax and ordinary global loads replacing removed legacy texture references; and B+tree's explicitly configured random query value. K-means therefore preserves the modeled logical accesses but may use a different cache path from its legacy texture implementation.

BFS uses a four-node chain. LavaMD uses one box with 100 particles. Particle Filter uses a 16×16 synthetic video, two frames and one particle. K-means uses one point and one cluster. B+tree uses one key and one lookup. The original application drivers and kernel implementations are retained; CPU preparation contracts used by static analysis are **not** substituted into these GPU runs.

Nsight Compute uses `--clock-control none` and `--cache-control none`; the script records GPU/tool versions and flags. It does not assume a fixed clock frequency. Keep the final download even if individual workloads fail.


In [ ]:
WORKLOADS = list(capture.CASES)  # All five; remove names to rerun a smaller selection.
status = capture.capture(RUN, WORKLOADS)
print(json.dumps(status, indent=2))
print('Measured cycles:', RUN / 'cycles.csv')
print('Per-workload success/failure:', RUN / 'status.json')
print('No timing-to-cycle estimates are produced.')


## Download the evidence
The ZIP contains cycle CSVs and `.ncu-rep` reports where available, `.nsys-rep`/SQLite traces where available, the matching Clang graphs, staged source hashes and compatibility changes, commands, stdout and errors. Upload the ZIP for source-versus-GPU DAG comparison. If counters are denied, use a GPU host with counter access for the cycle measurements; the notebook never invents them.


In [ ]:
archive = shutil.make_archive(str(RUN), 'zip', root_dir=RUN)
print(archive)
from google.colab import files
files.download(archive)
